# Assignment 3 – IoT Sensor Data Analysis

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('iot_sensor_data_raw.csv')
df.head()

,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status
0,2026-04-21 22:30:00,DEV_005,65.89,62.70,1007.96,2.22,46.51,Factory_A,Warning
1,2026-01-22 06:15:00,DEV_007,70.65,70.95,1021.85,2.73,57.04,Factory_A,Normal
2,2026-04-01 07:00:00,DEV_005,81.01,60.49,1009.82,2.15,11.65,Factory_A,Normal
3,2026-01-12 14:30:00,DEV_007,74.17,68.76,1001.42,2.03,17.36,Factory_C,Critical
4,2026-05-25 19:30:00,DEV_008,74.04,54.32,1014.34,NaN,27.04,Factory_C,Normal


## Part A – Data Preparation

In [2]:
# Q1 Load the dataset
df

,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status
0,2026-04-21 22:30:00,DEV_005,65.89,62.70,1007.96,2.22,46.51,Factory_A,Warning
1,2026-01-22 06:15:00,DEV_007,70.65,70.95,1021.85,2.73,57.04,Factory_A,Normal
2,2026-04-01 07:00:00,DEV_005,81.01,60.49,1009.82,2.15,11.65,Factory_A,Normal
3,2026-01-12 14:30:00,DEV_007,74.17,68.76,1001.42,2.03,17.36,Factory_C,Critical
4,2026-05-25 19:30:00,DEV_008,74.04,54.32,1014.34,NaN,27.04,Factory_C,Normal
...,...,...,...,...,...,...,...,...,...
19995,2026-04-28 13:00:00,DEV_001,70.77,51.92,997.89,2.41,20.87,Factory_A,Normal
19996,2026-05-05 15:00:00,DEV_004,70.75,51.21,1009.68,2.03,16.73,Factory_C,Warning
19997,2026-02-26 03:30:00,DEV_004,71.50,55.73,1046.63,2.60,88.53,Factory_C,Normal
19998,2026-01-09 23:00:00,DEV_004,70.65,63.28,1003.11,2.63,12.06,Factory_A,Normal


In [3]:
# Q2 Display dimensions and structure
print(df.shape)
df.info()

(20000, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Timestamp       20000 non-null  object 
 1   Device_ID       20000 non-null  object 
 2   Temperature     19701 non-null  float64
 3   Humidity        19700 non-null  float64
 4   Pressure        19700 non-null  float64
 5   Vibration       19702 non-null  float64
 6   Battery_Level   20000 non-null  float64
 7   Location        20000 non-null  object 
 8   Machine_Status  20000 non-null  object 
dtypes: float64(5), object(4)
memory usage: 1.4+ MB


In [4]:
# Q3 Convert Timestamp into datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Timestamp'].dtype

dtype('<M8[ns]')

In [5]:
# Q4 Check whether timestamps are correctly ordered
df['Timestamp'].is_monotonic_increasing

False

In [6]:
# Q5 Identify missing sensor readings
df.isnull().sum()

Timestamp           0
Device_ID           0
Temperature       299
Humidity          300
Pressure          300
Vibration         298
Battery_Level       0
Location            0
Machine_Status      0
dtype: int64

In [7]:
# Q6 Percentage of missing values for each sensor
(df.isnull().mean() * 100).round(2)

Timestamp         0.00
Device_ID         0.00
Temperature       1.50
Humidity          1.50
Pressure          1.50
Vibration         1.49
Battery_Level     0.00
Location          0.00
Machine_Status    0.00
dtype: float64

## Part B – Data Cleaning

In [8]:
# Q7 Handle missing sensor values using interpolation
sensor_cols = ['Temperature', 'Vibration', 'Battery_Level']
for col in sensor_cols:
    df[col] = df[col].interpolate(limit_direction='both')
df[sensor_cols].isnull().sum()

Temperature      0
Vibration        0
Battery_Level    0
dtype: int64

In [9]:
# Q8 Identify abnormal temperature readings (example threshold > 80°C)
abnormal_temp = df[df['Temperature'] > 80]
abnormal_temp.head()

,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status
2,2026-04-01 07:00:00,DEV_005,81.01,60.49,1009.82,2.15,11.65,Factory_A,Normal
18,2026-03-26 07:30:00,DEV_001,87.99,24.80,1020.36,2.06,32.27,Factory_A,Normal
23,2026-07-02 20:15:00,DEV_005,84.24,44.20,1008.92,0.79,79.37,Factory_B,Warning
25,2026-06-21 01:15:00,DEV_007,87.16,55.37,1001.03,2.16,43.13,Factory_B,Warning
41,2026-04-19 09:30:00,DEV_008,83.44,39.95,998.71,2.84,90.99,Factory_B,Normal


In [10]:
# Q9 Identify abnormal vibration readings (example threshold > 5)
abnormal_vibration = df[df['Vibration'] > 5]
abnormal_vibration.head()

,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status
471,2026-02-09 12:45:00,DEV_002,71.40,45.24,1016.22,5.300000,49.76,Factory_A,Normal
1420,2026-01-31 04:30:00,DEV_002,70.37,78.42,1044.17,11.262739,71.67,Factory_C,Normal
1972,2026-07-07 20:15:00,DEV_006,80.67,68.98,1016.54,17.874138,80.27,Factory_B,Normal
2011,2026-05-23 05:45:00,DEV_008,72.40,69.51,1013.55,11.338697,37.29,Factory_B,Warning
2325,2026-05-30 20:30:00,DEV_005,71.32,52.61,1021.26,13.188045,54.17,Factory_C,Normal


In [11]:
# Q10 Identify machines with critically low battery levels (<20)
critical_battery = df[df['Battery_Level'] < 20]
critical_battery.head()

,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status
2,2026-04-01 07:00:00,DEV_005,81.01,60.49,1009.82,2.15,11.65,Factory_A,Normal
3,2026-01-12 14:30:00,DEV_007,74.17,68.76,1001.42,2.03,17.36,Factory_C,Critical
33,2026-05-12 13:30:00,DEV_004,72.43,36.19,1016.77,1.53,14.33,Factory_C,Normal
38,2026-07-15 04:45:00,DEV_004,68.40,52.52,999.43,1.61,17.29,Factory_C,Normal
60,2026-03-20 16:30:00,DEV_006,69.73,53.25,1024.54,2.52,11.72,Factory_C,Critical


In [12]:
# Q11 Explain abnormal thresholds
print('Temperature > 80°C is treated as abnormal.')
print('Vibration > 5 is treated as abnormal.')
print('Battery < 20% is treated as critical.')

Temperature > 80°C is treated as abnormal.
Vibration > 5 is treated as abnormal.
Battery < 20% is treated as critical.


## Part C – Time-Series Analysis

In [13]:
# Q12 Average temperature by hour
df['Hour'] = df['Timestamp'].dt.hour
df.groupby('Hour')['Temperature'].mean()

Hour
0     70.076250
1     70.556449
2     69.872529
3     69.740815
4     69.849067
5     70.265227
6     70.264169
7     70.685270
8     70.394148
9     70.976106
10    69.956100
11    69.896311
12    69.986484
13    70.131852
14    69.821481
15    70.239221
16    70.013335
17    70.608696
18    70.228065
19    70.041887
20    70.168559
21    70.138049
22    71.166157
23    70.400673
Name: Temperature, dtype: float64

In [14]:
# Q13 Average temperature for each device
df.groupby('Device_ID')['Temperature'].mean()

Device_ID
DEV_001    70.264323
DEV_002    70.370172
DEV_003    69.943430
DEV_004    70.615241
DEV_005    70.126608
DEV_006    70.014669
DEV_007    70.145171
DEV_008    70.333818
Name: Temperature, dtype: float64

In [15]:
# Q14 Average vibration for each device
df.groupby('Device_ID')['Vibration'].mean()

Device_ID
DEV_001    2.552392
DEV_002    2.564100
DEV_003    2.538122
DEV_004    2.523990
DEV_005    2.495508
DEV_006    2.552213
DEV_007    2.517617
DEV_008    2.528933
Name: Vibration, dtype: float64

In [16]:
# Q15 Maximum temperature recorded by each device
df.groupby('Device_ID')['Temperature'].max()

Device_ID
DEV_001    159.746758
DEV_002    151.818550
DEV_003    157.603693
DEV_004    151.965273
DEV_005    152.974450
DEV_006    136.003203
DEV_007    153.877632
DEV_008    153.622123
Name: Temperature, dtype: float64

In [17]:
# Q16 Minimum battery level for each device
df.groupby('Device_ID')['Battery_Level'].min()

Device_ID
DEV_001    2.257059
DEV_002    2.154615
DEV_003    3.670604
DEV_004    8.514009
DEV_005    2.956738
DEV_006    2.201033
DEV_007    2.146086
DEV_008    2.011730
Name: Battery_Level, dtype: float64

In [18]:
# Q17 Device with the highest average vibration
avg_vibration = df.groupby('Device_ID')['Vibration'].mean()
avg_vibration.idxmax(), avg_vibration.max()

('DEV_002', 2.5640996658677073)

In [20]:
# Q18 Factory with the highest average temperature
avg_temp_factory = df.groupby('Location')['Temperature'].mean()
avg_temp_factory.idxmax(), avg_temp_factory.max()

('Factory_C', 70.28201298587999)

## Part D – Create New Variables

In [21]:
# Q19 Battery_Status
def battery_status(x):
    if x >= 50:
        return 'Healthy'
    elif x >= 20:
        return 'Moderate'
    else:
        return 'Critical'

df['Battery_Status'] = df['Battery_Level'].apply(battery_status)
df[['Battery_Level', 'Battery_Status']].head()

,Battery_Level,Battery_Status
0,46.51,Moderate
1,57.04,Healthy
2,11.65,Critical
3,17.36,Critical
4,27.04,Moderate


In [22]:
# Q20 Temperature_Status
def temperature_status(x):
    if x <= 60:
        return 'Normal'
    elif x <= 80:
        return 'Warning'
    else:
        return 'Critical'

df['Temperature_Status'] = df['Temperature'].apply(temperature_status)
df[['Temperature', 'Temperature_Status']].head()

,Temperature,Temperature_Status
0,65.89,Warning
1,70.65,Warning
2,81.01,Critical
3,74.17,Warning
4,74.04,Warning


In [23]:
# Q21 Vibration_Status
def vibration_status(x):
    if x <= 3:
        return 'Normal'
    elif x <= 5:
        return 'Warning'
    else:
        return 'Critical'

df['Vibration_Status'] = df['Vibration'].apply(vibration_status)
df[['Vibration', 'Vibration_Status']].head()

,Vibration,Vibration_Status
0,2.22,Normal
1,2.73,Normal
2,2.15,Normal
3,2.03,Normal
4,2.35,Normal


In [24]:
# Q22 Overall Machine_Health
def machine_health(row):
    statuses = [row['Battery_Status'], row['Temperature_Status'], row['Vibration_Status']]
    if 'Critical' in statuses:
        return 'Critical'
    elif 'Warning' in statuses:
        return 'Warning'
    else:
        return 'Normal'

df['Machine_Health'] = df.apply(machine_health, axis=1)
df[['Battery_Status', 'Temperature_Status', 'Vibration_Status', 'Machine_Health']].head()

,Battery_Status,Temperature_Status,Vibration_Status,Machine_Health
0,Moderate,Warning,Normal,Warning
1,Healthy,Warning,Normal,Warning
2,Critical,Critical,Normal,Critical
3,Critical,Warning,Normal,Critical
4,Moderate,Warning,Normal,Warning


## Part E – Advanced Analysis

In [25]:
# Q23 Devices with critical conditions more than five times
critical_counts = (
    df[df['Machine_Health'] == 'Critical']
    .groupby('Device_ID')
    .size()
)
critical_counts[critical_counts > 5]

Device_ID
DEV_001    532
DEV_002    543
DEV_003    498
DEV_004    547
DEV_005    549
DEV_006    509
DEV_007    496
DEV_008    533
dtype: int64

In [27]:
# Q24 Factory with the highest number of abnormal sensor readings
abnormal = df[
    (df['Temperature_Status'] != 'Normal') |
    (df['Vibration_Status'] != 'Normal')
]
abnormal.groupby('Location').size().sort_values(ascending=False)

Location
Factory_A    6201
Factory_C    6184
Factory_B    6104
dtype: int64

In [28]:
# Q25 Percentage of time each device operates under warning/critical conditions
status_mask = df['Machine_Health'].isin(['Warning', 'Critical'])
(
    status_mask.groupby(df['Device_ID']).mean() * 100
).round(2)

Device_ID
DEV_001    93.66
DEV_002    93.24
DEV_003    91.96
DEV_004    93.69
DEV_005    93.63
DEV_006    93.69
DEV_007    93.72
DEV_008    92.65
Name: Machine_Health, dtype: float64

In [29]:
# Q26 Device requiring the highest maintenance priority
priority = (
    df[df['Machine_Health'] == 'Critical']
    .groupby('Device_ID')
    .size()
)
priority.idxmax(), priority.max()

('DEV_005', 549)